# Mall Customer Segmentation with K-Means Clustering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BurakGuldal10/Machine-Learning-and-Data-Analysis-Projects/blob/main/Mall_Customer_Segmentation_KMeans.ipynb)

## 📌 Proje Özeti / Project Overview
Bu projede, bir alışveriş merkezinin (Mall) müşteri verileri kullanılarak denetimsiz öğrenme (Unsupervised Learning) yöntemi olan **K-Means Kümeleme** algoritması ile müşteri segmentasyonu gerçekleştirilmiştir. Müşterilerin **Yıllık Gelirleri (Annual Income)** ve **Harcama Skorları (Spending Score)** analiz edilerek optimum küme sayısı **Elbow (Dirsek)** ve **Silhouette Analizi** yöntemleriyle belirlenmiştir.

### 1. Veri Setinin İndirilmesi / Download Dataset

In [ ]:
import kagglehub
import os

# Kaggle üzerinden veri setini indirelim
path = kagglehub.dataset_download("vjchoudhary7/customer-segmentation-tutorial-in-python")
print("Path to dataset files:", path)

### 2. Kütüphanelerin Yüklenmesi / Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Görselleştirme ayarları
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Kütüphaneler hazır!")

### 3. Verinin Yüklenmesi ve İlk İnceleme / Data Loading & Inspection

In [ ]:
# CSV dosyasını okuyalım
csv_path = os.path.join(path, "Mall_Customers.csv")
df = pd.read_csv(csv_path)
print(f"Veri seti başarıyla yüklendi! Boyut: {df.shape}")
display(df.head())

### 4. Veri Bilgisi ve Eksik Değer Kontrolü / Data Info & Missing Values

In [ ]:
print("--- Veri Bilgisi ---")
df.info()

print("\n--- Eksik Değer Kontrolü ---")
print(df.isnull().sum())

print("\n--- Sayısal Dağılım Özeti ---")
display(df.describe())

### 5. Kümeleme Özelliklerinin Seçimi / Feature Selection

In [ ]:
# Müşterileri segmentlere ayırmak için Yıllık Gelir ve Harcama Skorunu seçiyoruz
X = df[['Annual Income (k$)', 'Spending Score (1-100)']].values
print(f"Kümeleme için seçilen veri matrisi şekli: {X.shape}")

### 6. Optimum Küme Sayısı: Elbow (Dirsek) Yöntemi / Elbow Method

In [ ]:
# 1'den 10'a kadar k değerleri için WCSS (Grup İçi Kareler Toplamı) hesaplayalım
wcss = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)

# Elbow grafiğini çizelim
plt.figure(figsize=(10, 5))
plt.plot(range(1, 11), wcss, marker='o', linestyle='-', color='indigo', linewidth=2, markersize=7)
plt.title('K-Means için Elbow (Dirsek) Grafiği', fontsize=14, fontweight='bold')
plt.xlabel('Küme Sayısı (k)', fontsize=12)
plt.ylabel('WCSS (Grup İçi Kareler Toplamı)', fontsize=12)
plt.xticks(range(1, 11))
plt.axvline(x=5, color='red', linestyle='--', label='Dirsek Noktası (k=5)')
plt.legend()
plt.tight_layout()
plt.show()

### 7. Silhouette Skoru Analizi / Silhouette Score Analysis

In [ ]:
print("=== Silhouette Skorları ===\n")
sil_scores = {}
for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X)
    sil_avg = silhouette_score(X, cluster_labels)
    sil_scores[k] = sil_avg
    print(f"Küme Sayısı {k} için Silhouette Skoru: {sil_avg:.4f}")

best_k = max(sil_scores, key=sil_scores.get)
print(f"\nEn yüksek Silhouette Skoruna sahip küme sayısı: k={best_k} (Skor: {sil_scores[best_k]:.4f})")

### 8. Nihai K-Means Modeli ve Kümelerin Görselleştirilmesi / Final Model & Visualization

In [ ]:
# Optimal k=5 ile nihai modeli eğitelim
optimal_k = 5
kmeans_final = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
y_kmeans = kmeans_final.fit_predict(X)
df['Cluster'] = y_kmeans

# Segmentleri görselleştirelim
plt.figure(figsize=(11, 7))
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']
cluster_names = [
    'Düşük Gelir - Yüksek Harcama (Özenli)',
    'Düşük Gelir - Düşük Harcama (Tasarruflu)',
    'Orta Gelir - Orta Harcama (Ortalama)',
    'Yüksek Gelir - Yüksek Harcama (Hedef Kitle)',
    'Yüksek Gelir - Düşük Harcama (Potansiyel)'
]

for i in range(optimal_k):
    plt.scatter(X[y_kmeans == i, 0], X[y_kmeans == i, 1], s=70, c=colors[i], label=f'Segment {i+1}: {cluster_names[i]}', alpha=0.8)

# Merkez noktaları (Centroids)
plt.scatter(kmeans_final.cluster_centers_[:, 0], kmeans_final.cluster_centers_[:, 1], 
            s=250, c='yellow', marker='*', edgecolor='black', label='Küme Merkezleri (Centroids)')

plt.title('Müşteri Segmentasyonu (K-Means k=5)', fontsize=15, fontweight='bold')
plt.xlabel('Yıllık Gelir (k$)', fontsize=12)
plt.ylabel('Harcama Skoru (1-100)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 9. Sonuç ve İş Önerileri / Business Insights & Recommendations

K-Means algoritması ile müşteriler 5 belirgin segmente ayrılmıştır:
1. **Yüksek Gelir - Yüksek Harcama (Hedef Kitle):** Premium ürünler, sadakat programları ve VIP kampanyaları için en ideal grup.
2. **Yüksek Gelir - Düşük Harcama (Potansiyel):** Satın alma gücü yüksek ancak harcama isteği düşük grup; kişiselleştirilmiş pazarlama stratejileri ile hedeflenmeli.
3. **Orta Gelir - Orta Harcama (Ortalama Müşteri):** Dengeli grup; genel kampanyalar ve popüler ürünlerle elde tutulmalı.
4. **Düşük Gelir - Yüksek Harcama (Özenli / Trend Takipçileri):** İndirimler, esnek ödeme seçenekleri ve taksit imkanları sunulmalı.
5. **Düşük Gelir - Düşük Harcama (Tasarruflu):** Temel ihtiyaç ve uygun fiyatlı ürünlerle etkileşime sokulabilir.